In [1]:
# Import libraries

import pandas as pd
import numpy as np

print("Libraries imported successfully. ✅")

Libraries imported successfully. ✅


Load integrated dataset

Why: We use the standardized dataset created in 01_dataset_integration.ipynb.

In [2]:
# Load the integrated dataset

df = pd.read_csv(
    "../data/processed/integrated_startup_data.csv"
)

print("Dataset loaded successfully. ✅")
print("Shape:", df.shape)

Dataset loaded successfully. ✅
Shape: (1383, 14)


Select labeled ML data

Why: Only the India AI dataset has reliable is_unicorn labels. We do not create fake labels for Funding records or train on the pre-selected Unicorn dataset.

In [3]:
# Select the labeled India AI records for ML

modeling_df = df[
    df["data_source"] == "India AI Dataset"
].copy()

print("Modeling dataset created. ✅")
print("Shape:", modeling_df.shape)

Modeling dataset created. ✅
Shape: (112, 14)


Check target distribution

Why: Confirm how many successful and non-successful startups are available for training.

In [4]:
# Check the target distribution

print(
    modeling_df["is_unicorn"].value_counts()
)

is_unicorn
False    101
True      11
Name: count, dtype: int64


Convert employee range

Why: Employee values may be ranges such as 200-500. ML models need numeric values, so we convert each range to an approximate midpoint.

In [5]:
# Convert employee ranges into numeric values

def employee_midpoint(value):
    if pd.isna(value):
        return np.nan

    value = str(value).strip()

    if "-" in value:
        parts = value.split("-")
        return (
            float(parts[0]) + float(parts[1])
        ) / 2

    if "+" in value:
        return float(
            value.replace("+", "")
        )

    return pd.to_numeric(
        value,
        errors="coerce"
    )

modeling_df["employees"] = (
    modeling_df["employees"]
    .apply(employee_midpoint)
)

print("Employee feature converted successfully. ✅")

Employee feature converted successfully. ✅


Create startup age

Why: Startup age can be an important business feature.

In [6]:
# Calculate startup age

current_year = 2026

modeling_df["startup_age"] = (
    current_year - modeling_df["founded_year"]
)

print("Startup age feature created. ✅")

Startup age feature created. ✅


Create log funding

Why: Funding is highly skewed. Log transformation reduces the effect of extremely large funding values.

In [7]:
# Create log-transformed funding

modeling_df["funding_log"] = np.log1p(
    modeling_df["funding_usd_millions"]
)

print("Funding log feature created. ✅")

Funding log feature created. ✅


Create log valuation

Why: Valuation can also have very large values, so log transformation makes the distribution easier for ML models to handle.

In [8]:
# Create log-transformed valuation

modeling_df["valuation_log"] = np.log1p(
    modeling_df["valuation_usd_millions"]
)

print("Valuation log feature created. ✅")

Valuation log feature created. ✅


Create funding per employee

Why: This measures funding relative to the size of the startup.

In [9]:
# Calculate funding per employee

modeling_df["funding_per_employee"] = (
    modeling_df["funding_usd_millions"]
    / modeling_df["employees"].replace(0, np.nan)
)

print("Funding per employee created. ✅")

Funding per employee created. ✅


Clean numeric infinity values

Why: Division can sometimes create infinite values. We convert them to missing values.

In [10]:
# Replace infinite values with missing values

modeling_df = modeling_df.replace(
    [np.inf, -np.inf],
    np.nan
)

print("Infinite values handled. ✅")

Infinite values handled. ✅


C:\Users\Rahul\AppData\Local\Temp\ipykernel_23144\2797818546.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  modeling_df = modeling_df.replace(


Check engineered features

Why: Verify that our new features were created correctly.

In [11]:
# Display the engineered features

print(
    modeling_df[
        [
            "company",
            "founded_year",
            "startup_age",
            "funding_usd_millions",
            "funding_log",
            "valuation_usd_millions",
            "valuation_log",
            "employees",
            "funding_per_employee",
            "is_unicorn"
        ]
    ].head()
)

         company  founded_year  startup_age  funding_usd_millions  \
38      Uniphore          2008           18                 987.0   
39  Eightfold AI          2016           10                 391.0   
40    Observe AI          2017            9                 214.0   
41         Pixis          2020            6                 209.0   
42     Darwinbox          2015           11                 110.0   

    funding_log  valuation_usd_millions  valuation_log  employees  \
38     6.895683                  2800.0       7.937732      750.0   
39     5.971262                  2100.0       7.650169      350.0   
40     5.370638                  1500.0       7.313887      350.0   
41     5.347108                  1000.0       6.908755      350.0   
42     4.709530                  1000.0       6.908755      750.0   

    funding_per_employee  is_unicorn  
38              1.316000        True  
39              1.117143        True  
40              0.611429       False  
41            

Check missing values

Why: Identify missing values before preprocessing and model training.

In [12]:
# Check missing values

print(
    modeling_df.isnull().sum()
)

company                     0
founded_year                0
sector                      0
subsector                   0
country                     0
city                        0
funding_usd_millions        0
valuation_usd_millions      0
employees                   0
startup_stage               0
profitable                112
is_unicorn                  0
data_source                 0
company_key                 0
startup_age                 0
funding_log                 0
valuation_log               0
funding_per_employee        0
dtype: int64


Select ML features

Why: We select only information that can reasonably be known about a startup. We exclude identifiers and the target.

In [13]:
# Select the features for machine learning

feature_columns = [
    "founded_year",
    "sector",
    "subsector",
    "country",
    "city",
    "funding_usd_millions",
    "valuation_usd_millions",
    "employees",
    "startup_stage",
    "startup_age",
    "funding_log",
    "valuation_log",
    "funding_per_employee"
]

X = modeling_df[feature_columns].copy()

y = modeling_df["is_unicorn"].astype(int)

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (112, 13)
Target shape: (112,)


Check features

Why: Final confirmation that the target is not accidentally included inside X

In [14]:
# Display final feature names

print("Features:")
print(X.columns.tolist())

print("\nTarget:")
print(y.name)

Features:
['founded_year', 'sector', 'subsector', 'country', 'city', 'funding_usd_millions', 'valuation_usd_millions', 'employees', 'startup_stage', 'startup_age', 'funding_log', 'valuation_log', 'funding_per_employee']

Target:
is_unicorn


Save feature-engineered dataset

Why: Save the prepared dataset so the training notebook can use exactly the same data.

In [15]:
# Save the feature-engineered dataset

modeling_df.to_csv(
    "../data/processed/feature_engineered_data.csv",
    index=False
)

print("Feature-engineered dataset saved successfully. ✅")

Feature-engineered dataset saved successfully. ✅


Final summary

Why: Confirm the final ML dataset before moving to model training.

In [16]:
# Display final feature engineering summary

print("========== FINAL SUMMARY ==========")

print("Rows:", len(modeling_df))
print("Features:", len(feature_columns))

print("\nTarget distribution:")
print(y.value_counts())

print("\nFeature columns:")
print(feature_columns)

========== FINAL SUMMARY ==========
Rows: 112
Features: 13

Target distribution:
is_unicorn
0    101
1     11
Name: count, dtype: int64

Feature columns:
['founded_year', 'sector', 'subsector', 'country', 'city', 'funding_usd_millions', 'valuation_usd_millions', 'employees', 'startup_stage', 'startup_age', 'funding_log', 'valuation_log', 'funding_per_employee']


In [17]:
# Check the current cleaned dataset sizes

funding_df = pd.read_csv(
    "../data/processed/funding_cleaned.csv"
)

india_df = pd.read_csv(
    "../data/processed/india_ai_startups_cleaned.csv"
)

unicorn_df = pd.read_csv(
    "../data/processed/unicorn_cleaned.csv"
)

print("Funding:", funding_df.shape)
print("India AI:", india_df.shape)
print("Unicorn:", unicorn_df.shape)

print(
    "\nTotal rows:",
    len(funding_df) + len(india_df) + len(unicorn_df)
)

Funding: (38, 27)
India AI: (112, 22)
Unicorn: (1233, 7)

Total rows: 1383
